# Support Ticket Router

TaskNest support tickets are classified by an OpenAI model and routed to Billing, Technical Support, or General Support. Add `OPENAI_API_KEY` to Colab Secrets, grant this notebook access, and run the cells in order.

## Setup

Install the dependencies, load the API key from Colab Secrets, and configure the classifier model.

In [ ]:
!pip -q install langgraph langchain-openai

In [ ]:
import os
from typing import Literal, NotRequired, TypedDict

from google.colab import userdata
from IPython.display import Image, display
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

MODEL_NAME = "gpt-6-luna"
llm = ChatOpenAI(model=MODEL_NAME, temperature=0, max_retries=0)

## State and nodes

The classifier selects one category. Each team node adds a fixed reply and a team name.

In [ ]:
Category = Literal["billing", "technical", "general"]


class TicketState(TypedDict):
    ticket: str
    category: NotRequired[Category]
    team: NotRequired[str]
    reply: NotRequired[str]


class TicketClassification(BaseModel):
    category: Category


CLASSIFICATION_INSTRUCTIONS = """You are a customer support ticket classifier for TaskNest, a to-do app.

Read the customer's ticket and classify it into one of three categories:

- billing: Payments, invoices, refunds, and subscriptions.
- technical: Bugs, crashes, error messages, and login problems.
- general: Feature requests, general questions, thank-you messages, and anything else.

Choose the category based on the main problem described in the ticket, not just keywords.

If a ticket mentions more than one topic, choose the category that best matches the main issue.

Return only the category."""

classifier = llm.with_structured_output(TicketClassification)


def classify(state: TicketState) -> dict[str, Category]:
    result = classifier.invoke(
        [
            ("system", CLASSIFICATION_INSTRUCTIONS),
            ("human", state["ticket"]),
        ]
    )
    return {"category": result.category}


def billing_team(state: TicketState) -> dict[str, str]:
    return {
        "team": "Billing",
        "reply": "Your ticket has been sent to our Billing team.",
    }


def tech_team(state: TicketState) -> dict[str, str]:
    return {
        "team": "Technical Support",
        "reply": "Your ticket has been sent to our Technical Support team.",
    }


def general_team(state: TicketState) -> dict[str, str]:
    return {
        "team": "General Support",
        "reply": "Your ticket has been sent to our General Support team.",
    }

## Build and display the graph

The conditional edge sends the ticket to the team for its category. The graph ends after that team adds its reply.

In [ ]:
def route(state: TicketState) -> Category:
    return state["category"]


builder = StateGraph(TicketState)
builder.add_node("classify", classify)
builder.add_node("billing_team", billing_team)
builder.add_node("tech_team", tech_team)
builder.add_node("general_team", general_team)

builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    route,
    {
        "billing": "billing_team",
        "technical": "tech_team",
        "general": "general_team",
    },
)
builder.add_edge("billing_team", END)
builder.add_edge("tech_team", END)
builder.add_edge("general_team", END)

graph = builder.compile()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

## Try sample tickets

Each ticket invokes the graph once. Compare the predicted category with the expected category, especially when a ticket mentions more than one topic.

In [ ]:
test_tickets = [
    ("I see two TaskNest charges for the same annual renewal. Can you reverse one?", "billing"),
    ("The app works fine. Where can I download a receipt for my subscription?", "billing"),
    ("The app closes every time I tap Pay. I want help fixing the crash so I can finish checking out.", "technical"),
    ("I reset my password, but signing in still sends me back to the login screen.", "technical"),
    ("Could you add a way to group my tasks by project and collapse each group?", "general"),
    ("Thanks for fixing the login error. Everything works again, and I just wanted to say thanks!", "general"),
]

for ticket, expected_category in test_tickets:
    result = graph.invoke({"ticket": ticket})
    print(f"Ticket: {result['ticket']}")
    print(f"Category: {result['category']} (expected: {expected_category})")
    print(f"Team: {result['team']}")
    print(f"Reply: {result['reply']}")
    print()